# Chapter 1 Preprocessing Runbook

This notebook is a thin orchestration and QC layer for the Chapter 1 preprocessing package in this repository.

- It reads standardized upstream ASIC artifacts from explicit artifact paths.
- It calls the repository package functions step by step.
- It displays compact QC summaries after each major stage.
- It writes final outputs into this repo's artifact directory.
- It reads runtime paths from a shared JSON config so the same settings can be reused by the CLI.


## Configuration

Edit the shared run config file before executing the notebook end to end. The default repo config points at the standardized ASIC artifact root in `icu-data-platform`.


In [1]:
from pathlib import Path

RUN_CONFIG_PATH = Path("config") / "ch1_run_config.json"


In [2]:
import sys
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "chapter1_mortality_decomposition").exists():
    if (PROJECT_ROOT.parent / "src" / "chapter1_mortality_decomposition").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repo root or the notebooks directory.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from chapter1_mortality_decomposition.artifacts import load_chapter1_inputs
from chapter1_mortality_decomposition.cohort import build_chapter1_cohort
from chapter1_mortality_decomposition.config import (
    build_chapter1_feature_set_definition,
    load_chapter1_feature_set_config,
)
from chapter1_mortality_decomposition.instances import build_chapter1_valid_instances
from chapter1_mortality_decomposition.labels import build_chapter1_proxy_horizon_labels
from chapter1_mortality_decomposition.model_ready import build_chapter1_model_ready_dataset
from chapter1_mortality_decomposition.pipeline import (
    Chapter1Dataset,
    Chapter1FeatureSetDataset,
    write_chapter1_dataset,
)
from chapter1_mortality_decomposition.run_config import load_chapter1_run_config

pd.set_option("display.max_colwidth", 160)

run_config = load_chapter1_run_config(PROJECT_ROOT / RUN_CONFIG_PATH)
config = run_config.to_chapter1_config()
feature_set_config = load_chapter1_feature_set_config(run_config.feature_set_config_path)
input_dir = run_config.input_dir
output_dir = run_config.output_dir

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Run config:** `{run_config.run_config_path}`"))
display(
    pd.DataFrame(
        [
            {"setting": "input_dir", "value": str(input_dir)},
            {"setting": "input_format", "value": run_config.input_format},
            {"setting": "output_dir", "value": str(output_dir)},
            {"setting": "output_format", "value": run_config.output_format},
            {
                "setting": "feature_set_config_path",
                "value": str(run_config.feature_set_config_path),
            },
            {"setting": "horizons_hours", "value": list(run_config.horizons_hours)},
            {
                "setting": "min_required_core_groups",
                "value": run_config.min_required_core_groups,
            },
            {
                "setting": "blocked_dynamic_features_artifact",
                "value": str(
                    input_dir / "blocked" / f"asic_8h_blocked_dynamic_features.{run_config.input_format}"
                ),
            },
        ]
    )
)


**Project root:** `/Users/joanameyer/repository/1-mortality-decomposition`

**Run config:** `/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_run_config.json`

,setting,value
0,input_dir,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized
1,input_format,csv
2,output_dir,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1
3,output_format,csv
4,feature_set_config_path,/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_feature_sets.json
5,horizons_hours,"[8, 16, 24, 48, 72]"
6,min_required_core_groups,3
7,blocked_dynamic_features_artifact,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/blocked/asic_8h_blocked_dynamic_features.csv


In [3]:
if not input_dir.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {input_dir}. Update {run_config.run_config_path}."
    )

inputs = load_chapter1_inputs(input_dir=input_dir, input_format=run_config.input_format)

artifact_overview = pd.DataFrame(
    [
        {"table": "static_harmonized", "rows": inputs.static_harmonized.shape[0], "columns": inputs.static_harmonized.shape[1]},
        {"table": "dynamic_harmonized", "rows": inputs.dynamic_harmonized.shape[0], "columns": inputs.dynamic_harmonized.shape[1]},
        {"table": "block_index", "rows": inputs.block_index.shape[0], "columns": inputs.block_index.shape[1]},
        {"table": "blocked_dynamic_features", "rows": inputs.blocked_dynamic_features.shape[0], "columns": inputs.blocked_dynamic_features.shape[1]},
        {"table": "stay_block_counts", "rows": inputs.stay_block_counts.shape[0], "columns": inputs.stay_block_counts.shape[1]},
    ]
)
display(Markdown("## Loaded Upstream Artifacts"))
display(artifact_overview)


## Loaded Upstream Artifacts

,table,rows,columns
0,static_harmonized,80,16
1,dynamic_harmonized,105165,107
2,block_index,3250,6
3,blocked_dynamic_features,3250,621
4,stay_block_counts,80,9


In [4]:
cohort = build_chapter1_cohort(
    static_harmonized=inputs.static_harmonized,
    dynamic_harmonized=inputs.dynamic_harmonized,
    stay_block_counts=inputs.stay_block_counts,
    config=config,
)

display(Markdown("## Cohort QC"))
display(pd.DataFrame([
    {
        "retained_hospitals": cohort.retained_hospitals.shape[0],
        "retained_stays": cohort.retained_stays.shape[0],
    }
]))
display(Markdown("### Site-level exclusions"))
display(cohort.site_eligibility[["hospital_id", "site_included_ch1", "usable_core_vital_group_count", "exclusion_reason"]])
display(Markdown("### Stay-level exclusions"))
display(cohort.stay_exclusion_summary_by_hospital)


## Cohort QC

,retained_hospitals,retained_stays
0,4,39


### Site-level exclusions

,hospital_id,site_included_ch1,usable_core_vital_group_count,exclusion_reason
0,asic_UK00,False,4,missing/unusable icu_mortality at site level
1,asic_UK01,False,0,missing/unusable icu_mortality at site level; insufficient core-vitals coverage
2,asic_UK02,True,4,<NA>
3,asic_UK03,False,3,missing/unusable icu_mortality at site level
4,asic_UK04,True,4,<NA>
5,asic_UK06,False,0,insufficient core-vitals coverage
6,asic_UK07,True,4,<NA>
7,asic_UK08,True,4,<NA>


### Stay-level exclusions

,hospital_id,before_site_level_exclusion,after_site_level_exclusion,after_no_dynamic_data_exclusion,after_missing_readmission_exclusion,after_readmission_flagged_exclusion,after_missing_icu_mortality_exclusion,final_retained_stays,excluded_no_dynamic_data_stays,excluded_missing_readmission_stays,excluded_readmission_flagged_stays,excluded_missing_icu_mortality_stays
0,asic_UK00,10,0,0,0,0,0,0,0,0,0,0
1,asic_UK01,10,0,0,0,0,0,0,0,0,0,0
2,asic_UK02,10,10,10,10,10,10,10,0,0,0,0
3,asic_UK03,10,0,0,0,0,0,0,0,0,0,0
4,asic_UK04,10,10,10,10,10,10,10,0,0,0,0
5,asic_UK06,10,0,0,0,0,0,0,0,0,0,0
6,asic_UK07,10,10,10,10,9,9,9,0,0,1,0
7,asic_UK08,10,10,10,10,10,10,10,0,0,0,0


In [5]:
feature_set_definition, feature_set_validation_summary = build_chapter1_feature_set_definition(
    inputs.blocked_dynamic_features,
    retained_stays=cohort.retained_stays,
    config=config,
    feature_set_config=feature_set_config,
)

display(Markdown("## Feature-set Configuration and Validation"))
display(feature_set_validation_summary[[
    "feature_set_name",
    "primary_feature_count",
    "extended_only_feature_count",
    "total_extended_feature_count",
    "available_in_blocked_schema_count",
    "missing_from_blocked_schema_count",
    "absent_from_retained_data_count",
    "missing_from_blocked_schema_features",
    "absent_from_retained_data_features",
]])


## Feature-set Configuration and Validation

,feature_set_name,primary_feature_count,extended_only_feature_count,total_extended_feature_count,available_in_blocked_schema_count,missing_from_blocked_schema_count,absent_from_retained_data_count,missing_from_blocked_schema_features,absent_from_retained_data_features
0,primary,31,15,46,31,0,0,[],[]
1,extended,31,15,46,46,0,0,[],[]


In [6]:
feature_set_runs = {}

for feature_set_name in feature_set_config.feature_sets:
    display(Markdown(f"## {feature_set_name.title()} Feature Set"))
    feature_set_subset = feature_set_definition[
        feature_set_definition["feature_set_name"].eq(feature_set_name)
    ].reset_index(drop=True)

    valid_instances = build_chapter1_valid_instances(
        retained_cohort=cohort.table,
        block_index=inputs.block_index,
        blocked_dynamic_features=inputs.blocked_dynamic_features,
        feature_set_definition=feature_set_subset,
        config=config,
    )
    labels = build_chapter1_proxy_horizon_labels(
        valid_instances=valid_instances.valid_instances,
        retained_cohort=cohort.table,
    )
    model_ready = build_chapter1_model_ready_dataset(
        usable_labels=labels.usable_labels,
        blocked_dynamic_features=inputs.blocked_dynamic_features,
        feature_set_definition=feature_set_subset,
        feature_set_name=feature_set_name,
    )

    feature_set_runs[feature_set_name] = Chapter1FeatureSetDataset(
        feature_set_name=feature_set_name,
        valid_instances=valid_instances,
        labels=labels,
        model_ready=model_ready,
    )

    display(Markdown("### Valid prediction-instance construction"))
    display(valid_instances.counts_by_horizon)
    display(Markdown("### Proxy horizon label generation"))
    display(labels.summary_by_horizon)
    display(labels.unlabeled_reason_summary)
    display(Markdown("### Model-ready dataset creation"))
    display(model_ready.readiness_summary)


## Primary Feature Set

### Valid prediction-instance construction

,horizon_h,candidate_instances,valid_instances,excluded_instances
0,8,1724,1722,2
1,16,1724,1722,2
2,24,1724,1722,2
3,48,1724,1722,2
4,72,1724,1722,2


### Proxy horizon label generation

,horizon_h,total_valid_prediction_instances,labelable_instances,positive_labels,negative_labels,unlabeled_instances
0,8,1722,1291,11,1280,431
1,16,1722,1274,22,1252,448
2,24,1722,1257,33,1224,465
3,48,1722,1210,63,1147,512
4,72,1722,1163,87,1076,559


...

### Proxy horizon label generation

,horizon_h,total_valid_prediction_instances,labelable_instances,positive_labels,negative_labels,unlabeled_instances
0,8,1722,1291,11,1280,431
1,16,1722,1274,22,1252,448
2,24,1722,1257,33,1224,465
3,48,1722,1210,63,1147,512
4,72,1722,1163,87,1076,559


,horizon_h,unlabeled_reason,instance_count
0,8,missing_required_fields,0
1,8,prediction_time_not_strictly_before_proxy_end,0
2,8,survivor_without_full_horizon_observation,27
3,8,non_survivor_proxy_end_not_within_horizon,404
4,8,unsupported_icu_mortality_code,0
5,16,missing_required_fields,0
6,16,prediction_time_not_strictly_before_proxy_end,0
7,16,survivor_without_full_horizon_observation,55
8,16,non_survivor_proxy_end_not_within_horizon,393
9,16,unsupported_icu_mortality_code,0


### Model-ready dataset creation

,feature_set_name,metric,value,note
0,extended,model_ready_rows_total,6195,Rows after valid-instance selection and label availability filtering.
1,extended,configured_base_features_total,46,Configured base features for this Chapter 1 feature set.
2,extended,selected_feature_columns_total,276,Blocked dynamic feature columns selected by the Chapter 1 feature config.
3,extended,distinct_horizons_total,5,Configured prediction horizons represented in the model-ready table.
4,extended,configured_base_features_missing_from_blocked_schema,,Configured base features absent from the current blocked schema.
5,extended,configured_base_features_absent_from_retained_data,,Configured base features absent from current retained-hospital ASIC data.
6,extended,label_definition_in_model_ready_dataset,proxy_within_horizon_icu_mortality_v1,Current model-ready rows use the explicit ASIC proxy within-horizon label definition.


In [7]:
dataset = Chapter1Dataset(
    cohort=cohort,
    feature_set_definition=feature_set_definition,
    feature_set_validation_summary=feature_set_validation_summary,
    feature_sets=feature_set_runs,
)

output_paths = write_chapter1_dataset(
    dataset,
    output_dir=output_dir,
    output_format=run_config.output_format,
)

display(Markdown("## Outputs Written"))
output_manifest = pd.DataFrame(
    {
        "artifact_key": list(output_paths.keys()),
        "path": [str(path) for path in output_paths.values()],
    }
)
display(output_manifest)

display(Markdown("## Final Quick QC"))
final_qc_rows = []
for feature_set_name, feature_set_dataset in feature_set_runs.items():
    readiness = feature_set_dataset.model_ready.readiness_summary.set_index("metric")
    final_qc_rows.append(
        {
            "feature_set_name": feature_set_name,
            "valid_instances_total": int(feature_set_dataset.valid_instances.valid_instances.shape[0]),
            "labelable_instances_total": int(feature_set_dataset.labels.usable_labels.shape[0]),
            "model_ready_rows_total": int(feature_set_dataset.model_ready.table.shape[0]),
            "selected_feature_columns_total": readiness.at["selected_feature_columns_total", "value"],
        }
    )
display(pd.DataFrame(final_qc_rows))

print(f"Saved {len(output_paths)} output tables to {output_dir}")


## Outputs Written

,artifact_key,path
0,cohort_chapter1_notes,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_notes.csv
1,cohort_chapter1_core_vital_group_coverage,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_core_vital_group_coverage.csv
2,cohort_chapter1_site_eligibility,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_eligibility.csv
3,cohort_chapter1_site_counts_summary,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_counts_summary.csv
4,cohort_chapter1_stay_exclusions,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusions.csv
5,cohort_chapter1_stay_exclusion_summary_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusion_summary_by_hospital.csv
6,cohort_chapter1_counts_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_counts_by_hospital.csv
7,cohort_chapter1_retained_hospitals,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_hospitals.csv
8,cohort_chapter1_retained_stays,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stays.csv
9,cohort_chapter1_retained_stay_table,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stay_table.csv


## Final Quick QC

,feature_set_name,valid_instances_total,labelable_instances_total,model_ready_rows_total,selected_feature_columns_total
0,primary,8610,6195,6195,186
1,extended,8610,6195,6195,276


Saved 36 output tables to /Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1


In [9]:
display(inputs.blocked_dynamic_features)

,stay_id_global,hospital_id,block_index,block_start_h,block_end_h,prediction_time_h,dynamic_row_count,non_missing_measurements_in_block,observed_variables_in_block,heart_rate_obs_count,...,fludrocortisone_po_bolus_median,fludrocortisone_po_bolus_min,fludrocortisone_po_bolus_max,fludrocortisone_po_bolus_last,sofa_obs_count,sofa_mean,sofa_median,sofa_min,sofa_max,sofa_last
0,asic_UK00_9990,asic_UK00,0,0,8,8,32,208,31,22,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
1,asic_UK00_9990,asic_UK00,1,8,16,16,32,331,40,17,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2,asic_UK00_9990,asic_UK00,2,16,24,24,32,471,34,32,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3,asic_UK00_9990,asic_UK00,3,24,32,32,32,303,20,29,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
4,asic_UK00_9990,asic_UK00,4,32,40,40,32,376,38,31,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3245,asic_UK08_9998,asic_UK08,46,368,376,376,32,576,48,31,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3246,asic_UK08_9998,asic_UK08,47,376,384,384,32,284,33,15,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3247,asic_UK08_9999,asic_UK08,0,0,8,8,31,154,29,13,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3248,asic_UK08_9999,asic_UK08,1,8,16,16,32,318,26,32,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
